# Feature Selection Strategy Implementation

**Pipeline order:** run this notebook first, then `baseline.ipynb` (optional), then any modeling notebook, then `modeling_compare_all.ipynb` to compare runs.

### Covered Workflow
- Stage 1: Broad univariate filtering (500 -> ~80)
- Stage 2: Multicollinearity and L1 regularization (80 -> ~25)
- Stage 3: CV drop-column ranking of Stage 2 features (business scorer, no full-data RFECV)

### Included Components
- Data loading and sanity checks
- Custom business scorer aligned with project metric
- Stage-by-stage feature reduction and validation
- Drop-column CV ranking of Stage 2 features
- Export of the Stage 2 ranking artifact consumed by the modeling notebook

## Key Parameters

| Parameter | Default | Description |
|---|---|---|
| custom_scorer | Required | Scoring function: scorer(estimator, X, y) -> float |
| stage1_n_features | 80 | Target features after Stage 1 |
| stage2_n_features | None | Optional strict target after Stage 2 |
| stage2_n_features_band | (20, 30) | Preferred approximate Stage 2 band (data-driven final count) |
| correlation_threshold | 0.85 | Correlation pruning threshold |
| vif_threshold | 5.0 | VIF filtering threshold |
| variance_threshold | 0.01 | Variance threshold (Stage 1) |
| ranking_cv_folds | 5 | CV folds for drop-column business ranking |

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from cost_effective.dataset import (
    MulticollinearityFilter,
    UnivariateFeatureFilter,
    find_project_root,
    get_classifier,
    load_training_data,
)
from cost_effective.models import (
    build_top_k_feature_sets,
    evaluate_feature_sets,
    rank_features_drop_column_cv,
)

np.random.seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

PROJECT_ROOT = find_project_root()
DATA_PATH = PROJECT_ROOT / "data"
OUTPUTS_PATH = PROJECT_ROOT / "outputs"

In [2]:
X_all, y = load_training_data(DATA_PATH)

print(f"✓ Loaded training data: {X_all.shape}")
print(f"  Class distribution: {y.value_counts().to_dict()}")

✓ Loaded training data: (5000, 500)
  Class distribution: {0: 2512, 1: 2488}


In [3]:
# Stage 1 and Stage 2
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*")
    warnings.filterwarnings("ignore", message=".*Inconsistent values: penalty=l1.*")

    print("=" * 70)
    print("STAGE 1: UNIVARIATE FILTERING (500 → 80)")
    print("=" * 70 + "\n")

    stage1_filter = UnivariateFeatureFilter(variance_threshold=0.01)
    X_stage1, stage1_features = stage1_filter.fit_transform(X_all, y, n_features=80)

    print()
    stage2_filter = MulticollinearityFilter(
        correlation_threshold=0.85,
        vif_threshold=5.0,
        random_state=42,
    )
    X_stage2, stage2_features = stage2_filter.fit_transform(
        X_stage1,
        y,
        n_features_target=None,
        n_features_band=(20, 30),
    )

print(f"✓ Stage 1 output: {len(stage1_features)} features")
print(f"✓ Stage 2 output: {len(stage2_features)} features")

STAGE 1: UNIVARIATE FILTERING (500 → 80)

[Stage 1a] Variance Threshold: 500 → 500 features
[Stage 1b] Mutual Information computed for 500 features
  Top 10 MI scores:
     feature  mi_score
10    var_10  0.029785
456  var_456  0.021367
470  var_470  0.020561
312  var_312  0.018987
159  var_159  0.018163
415  var_415  0.018163
175  var_175  0.018041
254  var_254  0.017258
4      var_4  0.017163
379  var_379  0.017061

[Stage 1c] LightGBM Importance computed
  Top 10 LGBM Importances:
     feature  lgbm_importance
214  var_214               94
379  var_379               94
341  var_341               81
254  var_254               73
190  var_190               71
116  var_116               67
159  var_159               64
226  var_226               61
389  var_389               60
482  var_482               58

[Stage 1 Output] Selected 80 features out of 500

STAGE 2: MULTICOLLINEARITY & L1 REGULARIZATION FILTERING

[Stage 2a] Correlation Pruning (threshold=0.85):
  High-correlation pair

In [4]:
# Stage 3: drop-column CV ranking (business scorer, avoids full-data RFECV leakage)
print("=" * 70)
print("STAGE 3: DROP-COLUMN CV RANKING (BUSINESS SCORER)")
print("=" * 70)

RANKING_CV_FOLDS = 5
ranked_stage2_features = rank_features_drop_column_cv(
    X_stage2,
    y,
    estimator_factory=get_classifier,
    cv=RANKING_CV_FOLDS,
)

print("\n✓ Drop-column ranking completed")
print(ranked_stage2_features.head(10).to_string(index=False))

STAGE 3: DROP-COLUMN CV RANKING (BUSINESS SCORER)

✓ Drop-column ranking completed
feature  cv_score_if_dropped  delta  order
var_389              -2497.0  187.0      1
var_308              -2496.0  188.0      2
 var_93              -2496.0  188.0      3
 var_87              -2492.0  192.0      4
var_254              -2490.0  194.0      5
var_365              -2488.0  196.0      6
var_420              -2487.0  197.0      7
var_463              -2487.0  197.0      8
var_448              -2485.0  199.0      9
var_190              -2483.0  201.0     10


In [5]:
ranked_stage2_features.to_csv(OUTPUTS_PATH / "feature_selection_results.csv", index=False)

In [6]:
candidate_feature_sizes = (1, 3, 5, 8, 10, 15, 20, len(stage2_features))
feature_set_candidates = build_top_k_feature_sets(
    ranked_stage2_features["feature"],
    sizes=candidate_feature_sizes,
)

stage3_feature_set_scores = evaluate_feature_sets(
    X_stage2,
    y,
    feature_set_candidates,
    estimator_factory=get_classifier,
    cv=5,
)
stage3_feature_set_scores.to_csv(OUTPUTS_PATH / "stage3_feature_set_scores.csv", index=False)

stage3_feature_set_scores

,feature_set_name,feature_count,cv_score_mean,cv_score_std,f1_score,roc_auc_score,business_score_no_var_penalty,features
0,top_01,1,2301.0,24.779023,0.667868,0.547226,2501.0,[var_389]
1,top_03,3,1891.0,32.155870,0.666970,0.550985,2491.0,"[var_389, var_308, var_93]"
2,top_05,5,1490.0,23.874673,0.666908,0.589741,2490.0,"[var_389, var_308, var_93, var_87, var_254]"
3,top_08,8,898.0,30.757113,0.667579,0.583463,2498.0,"[var_389, var_308, var_93, var_87, var_254, va..."
4,top_10,10,486.0,17.146428,0.666538,0.616666,2486.0,"[var_389, var_308, var_93, var_87, var_254, va..."
5,top_15,15,-484.0,21.307276,0.669326,0.619034,2516.0,"[var_389, var_308, var_93, var_87, var_254, va..."
6,top_20,20,-1472.0,48.744230,0.670652,0.624620,2528.0,"[var_389, var_308, var_93, var_87, var_254, va..."
7,top_26,26,-2684.0,43.977267,0.669447,0.627023,2516.0,"[var_389, var_308, var_93, var_87, var_254, va..."
